***Data Preprocessing Steps:-***

In [ ]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()
warnings.filterwarnings('ignore')

In [ ]:
company = pd.read_csv("companies.csv")
company

In [ ]:
company.head()

In [ ]:
company.describe()

In [ ]:
company.columns

## A. company Cleaning
    1. Delete irrelevant & redundant information.
    2. Remove noise or unreliable company (missing values and outliers).
    
### 1. Delete irrelevant and redundant information
     a. Delete 'region','city','state_code' as they provide too much of granularity.
     b. Delete 'id', 'Unnamed: 0.1', 'entity_type', 'entity_id', 'parent_id', 'created_by',
       'created_at', 'updated_at' as they are redundant.
     c. Delete 'domain', 'homepage_url', 'twitter_username', 'logo_url', 'logo_width', 'logo_height',           
        'short_description', 'description', 'overview','tag_list', 'name', 'normalized_name', 'permalink',    
        'invested_companies' as they are irrelevant features.
     d. Delete duplicate values if any.
     e. Delete those which has more than 98% of null values.
     
### 2. Remove noise or unreliable company (missing values and outliers)
     a. Delete instances with missing values for 'status', 'country_code', 'category_code' and 'founded_at'.
     b. Delete outliers for 'funding_total_usd' and 'funding_rounds'.
     c. Delete contradictory (mutually opposed or inconsistent company).

#### 1.a. Delete 'region','city' as they provide too much of granularity.    

In [ ]:
#Type your code here!
company = company.drop(['region','city'],axis=1)

#### 1.b. Delete 'id', 'Unnamed: 0.1', 'entity_type', 'entity_id', 'parent_id', 'created_by', 'created_at', 'updated_at' as they are redundant.

In [ ]:
#Type your code here!
company = company.drop(['id', 'Unnamed: 0.1', 'entity_type', 'entity_id', 'parent_id', 'created_by', 'created_at', 'updated_at'], axis=1)

#### 1.c. Delete 'domain', 'homepage_url', 'twitter_username', 'logo_url', 'logo_width', 'logo_height',  'short_description',    'description',  'overview','tag_list', 'name', 'normalized_name', 'permalink', 'invested_companies' as they are irrelevant features.

In [ ]:
#Type your code here!
company = company.drop(['domain', 'homepage_url', 'twitter_username', 'logo_url', 'logo_width', 'logo_height',  'short_description',    'description',  'overview','tag_list', 'name', 'normalized_name', 'permalink', 'invested_companies'], axis =1)

In [ ]:
company.head()

#### 1.d. Delete duplicate values if found any.

In [ ]:
# Delete duplicate values if found any.
#Type your code here!
company.duplicated().sum()  

In [ ]:
# Let's delete all the duplicate values
#Type your code here!
company.drop_duplicates(inplace=True)   

In [ ]:
# check if any left
#Type your code here!
company.duplicated().sum()

#### 1.e. Delete those which has more than 98% of null values.

In [ ]:
company

In [ ]:
# # Since we can see only nan values so et's check how much of ros has nan values.
#Type your code here!
company.isnull().sum()

In [ ]:
# Since we can see it has more than 96% of null values, it would not make sense to impute these data. So, let's drop it.
#Type your code here!
# Calculate the threshold for 96% null values
threshold = len(company) * 0.96

# Drop columns with more than 96% null values except for 'closed_at'
columns_to_drop = company.columns[company.isnull().sum() > threshold]
columns_to_drop = columns_to_drop.drop('closed_at', errors='ignore')

# Drop the identified columns from the DataFrame
company = company.drop(columns=columns_to_drop)


In [ ]:
company.isna().sum()

#### 2.a. Delete instances with missing values for 'status', 'country_code', 'category_code' and 'founded_at'.
    (Since these are the type of data where adding value via imputation will create wrong pattern only)

In [ ]:
#Type your code here!
company.dropna(subset =['status','country_code', 'category_code','founded_at'], inplace=True)

In [ ]:
# Since we can see only nan values so et's check how much of rows has nan values.
#Type your code here!
company.isnull().sum()

In [ ]:
company

#### 2.b. Delete outliers for 'funding_total_usd' and 'funding_rounds'.

In [ ]:
#Type your code here!

### Summary:
If you can see the outlier in both 'funding_total_usd' and 'funding_rounds'. So, let's find them and drop it.

    1. Find the IQR (Interquartile Range)
    2. Find the upper and lower limit
    3. Find outliers
    4. Drop them
    5. Compare the plots after trimming


#### 2.b.1. Find the IQR

In [ ]:
# For funding_total_usd
Q1 = company['funding_total_usd'].quantile(0.25)
Q3 = company['funding_total_usd'].quantile(0.75)
IQR = Q3 - Q1
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR
# company = company[(company['funding_total_usd'] >= lower_limit) & (company['funding_total_usd'] <= upper_limit)]

# For funding_rounds
Q1 = company['funding_rounds'].quantile(0.25)
Q3 = company['funding_rounds'].quantile(0.75)
IQR = Q3 - Q1
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR
# company = company[(company['funding_rounds'] >= lower_limit) & (company['funding_rounds'] <= upper_limit)]


#### 2.b.1.  Find outliers

In [ ]:
# For funding_total_usd
outliers_total_usd = company[(company['funding_total_usd'] < lower_limit) | (company['funding_total_usd'] > upper_limit)]

# For funding_rounds
outliers_rounds = company[(company['funding_rounds'] < lower_limit) | (company['funding_rounds'] > upper_limit)]

outliers_total_usd, outliers_rounds

#### 2.b.1. Drop the outliers

In [ ]:
# Drop outliers from company DataFrame
company = company[~company.index.isin(outliers_rounds.index)]
company = company[~company.index.isin(outliers_total_usd.index)]


In [ ]:
company

#### 2.c. Delete contradictory (mutually opposed or inconsistent data).


In [ ]:
# Since we have not imputed the datasets in closed_at yet, we will check it later on.

# B. Date Transformation
    It can be divided into two successive phases.
   ## 1. Changes in original data
        a. Convert founded_at, closed_at, first_funded_at, last_funding_at, first_milestone_at ,
           last_milestone_at to years.
        b. Generalize the categorical data i.e. category_code, status and category_code.
   ## 2. Create new variables
        a. Create new feature isClosed from closed_at and status.
        b. Create new feature 'active_days'

#### 1.a. Convert founded_at, closed_at, first_funded_at, last_funding_at, first_milestone_at , last_milestone_at to years.

In [ ]:
# For founded_at
company['founded_at'] = pd.to_datetime(company['founded_at']).dt.year

# For closed_at
company['closed_at'] = pd.to_datetime(company['closed_at']).dt.year

# For first_funded_at
company['first_funding_at'] = pd.to_datetime(company['first_funding_at']).dt.year

# For last_funding_at
company['last_funding_at'] = pd.to_datetime(company['last_funding_at']).dt.year

# For first_milestone_at
company['first_milestone_at'] = pd.to_datetime(company['first_milestone_at']).dt.year

# For last_milestone_at
company['last_milestone_at'] = pd.to_datetime(company['last_milestone_at']).dt.year

company

 #### 1.b. Generalize the categorical data i.e. category_code and  country_code

In [ ]:
# 1. category_code
#Type your code here!
company['category_code'].value_counts()

In [ ]:
# Since there are 42 categories, one-hot encoding which is going to create a lot of columns so
# Lets Check the repetition of value in ascending order and keep the first 10 values and name
# remaining one as other.
other_categories = company['category_code'].value_counts()[10:].index
company['category_code'] = company['category_code'].replace(to_replace=other_categories, value='other')
company['category_code'].value_counts().sort_values(ascending=False)
#Type your code here!

In [ ]:
# One-hot encoding to category_code
company = pd.get_dummies(company, columns=['category_code'], drop_first=True)
#Type your code here!

In [ ]:
# Since, We've added the encoded category_columns, let's delete original category_code
#Type your code here!

### Let's encode 'country' column now.

In [ ]:
# 1. country_code
company['country_code'].value_counts()

#Type your code here!

In [ ]:

# Since there are 161 categories, one-hot encoding which is going to create a lot of columns so
# Lets Check the repetition of value in ascending order and keep the first 10 values and name
# remaining one as other.
other_countries = company['country_code'].value_counts()[10:].index
company['country_code'] = company['country_code'].replace(to_replace=other_countries, value='other')
company['country_code'].value_counts().sort_values(ascending=False)
#Type your code here!


In [ ]:
# One-hot encoding to category_code
company = pd.get_dummies(company, columns=['country_code'], drop_first=True)
#Type your code here!

In [ ]:
# Since, We've added the encoded country_code , let's delete original category_code
#Type your code here!
company.head()

### 2. Create new variables¶
    a. Create new feature isClosed from closed_at and status.
    b. Create new feature 'active_days'

#### 2.a. Create new feature isClosed from closed_at and status.
     - if the value in status is 'operating' or 'ipo', Let's put 1.
     - Where as if the value is 'acquired' or 'closed', let's put 0.

In [ ]:
#Type your code here!
company['status'] = company['status'].replace('operating',1)
company['status'] = company['status'].replace('ipo',1)
company['status'] = company['status'].replace('closed',0)
company['status'] = company['status'].replace('acquired',0)

In [ ]:
#Type your code here!
# Create new feature isClosed from closed_at and status
company['isClosed'] = company.apply(lambda row: 1 if row['status'] == 'closed' else 0, axis=1)

In [ ]:
#Type your code here!
company.head()

#### 2.b. Create active_days
     i. Replacing values:
         -  if the value in status is 'operating' or 'ipo' in closed_at, Let's put 2021.
         - Where as if the value is 'acquired' or 'closed', let's put 0.
     ii. Subtract founded_date from closed_date, and calculate age in days (After calculating active days,
         check contradictory issues we didn't check it before).
     iii. Then, delete the closed_at column.

In [ ]:
#Type your code here!
for i in company['status']:
    if (i == 'operating' or 'ipo'):
        company['closed_at'].fillna(2021,inplace = True)
    elif (i == 'accquired' or 'closed'):
        company['closed_at'].fillna(0, inplace = True)

##### 2.b.i  Replacing the values in closed_at column
   - if the value in status is 'operating' or 'ipo' in closed_at, Let's put 2021.
   - Where as if the value is 'acquired' or 'closed', let's put 0.

##### 2.b.ii Subtract founded_date from closed_date, and calculate age in days (After calculating active days, check contradictory issues we didn't check it before.)

In [ ]:
#Type your code here!
# Subtract founded_date from closed_date, and calculate age in days
company['age_in_days'] = (company['closed_at'] - company['founded_at'])

#### 2.b.iii. Then, delete the closed_at column.

In [ ]:
#Type your code here!
company.drop(['closed_at'], axis=1, inplace=True)

### Let's work on target variabe now.

In [ ]:
#Type your code here!

In [ ]:
#Type your code here!

In [ ]:
#Type your code here!

In [ ]:
#Type your code here!

### Remove the null vaues with the mean value in 'Numerical Data'

In [ ]:
company

In [ ]:
#Type your code here!

#Type your code here!
company.isna().sum()    

In [ ]:
# Fill null values with mean in numerical columns
company.fillna(company.mean(), inplace=True)

In [ ]:
company.isna().sum()

In [ ]:
company.head()

In [ ]:
company.drop(['state_code','funding_total_usd'], axis=1, inplace=True)

In [ ]:
company.shape

In [ ]:
# Final null check on data
#Type your code here!
company.isna().sum()    

In [ ]:
#Finally Save cleaned Data
#Type your code here!

company.to_csv('cleaned_companies.csv', index=False)